# Zoom-in: two dark ORFs on a ColE1-like plasmid

Two families from `dark_families_90pct.ipynb` §2, picked out because they were the most
*habitat*-spread entries in that top 20 while sitting on only a handful of lineages:

| | representative | length | plasmids | lineages | habitats | Pfam |
|---|---|---:|---:|---:|---:|---|
| **ORF-A** | `GenBank_CP063716.1|6` | 155 aa | 400 | 5 | 23 | none |
| **ORF-B** | `IMGPR_plasmid_2609460067_000008|1` | 132 aa | 355 | 5 | 21 | none |

Both are unannotatable by PlasAnn and have no Pfam-A match at `--cut_ga`. This notebook asks
what they are, and the answer turns out to be a single object rather than two.

Clustering is the 90% run (`scripts/recluster_sweep.sbatch`,
`--min-seq-id 0.9 -c 0.8 --cov-mode 0 --cluster-reassign 1`), so "family" here means
near-identical protein.

In [ ]:
# Zoom-in on two dark ORF families that surfaced in dark_families_90pct.ipynb §2:
#     A = GenBank_CP063716.1|6              155 aa rep, 400 plasmids, 5 lineages, 23 habitats
#     B = IMGPR_plasmid_2609460067_000008|1 132 aa rep, 355 plasmids, 5 lineages, 21 habitats
# Both Pfam-negative at --cut_ga, both on ~5.2 kb carriers, and both far more habitat-spread
# than anything else in that top 20. This notebook asks what they actually are.
#
# Clustering is the 90% run (scripts/recluster_sweep.sbatch:
# --min-seq-id 0.9 -c 0.8 --cov-mode 0 --cluster-reassign 1), so "family" here means
# near-identical protein, not distant homolog.
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RC, PP = ROOT / "data" / "dark_orf_run" / "recluster", ROOT / "data" / "plasmidscope_primary"

S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"


def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)


A, B = "GenBank_CP063716.1|6", "IMGPR_plasmid_2609460067_000008|1"
LAB = {A: "ORF-A", B: "ORF-B"}

CL = pd.read_csv(RC / "dark90_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
CL["pl"] = CL["member"].str.rsplit("|", n=1).str[0]
CL["idx"] = CL["member"].str.rsplit("|", n=1).str[1].astype(int)


def read_faa(p):
    out, name, cur = {}, None, []
    for line in open(p):
        if line[0] == ">":
            if name is not None:
                out[name] = "".join(cur)
            name, cur = line[1:].strip().split()[0], []
        else:
            cur.append(line.strip())
    out[name] = "".join(cur)
    return out


SEQ = read_faa(ROOT / "data" / "dark_orf_run" / "dark_orfs.faa")
MET = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False).set_index("plasmid_id")
ORG = pd.read_csv(PP / "working_set.tsv", sep="\t", low_memory=False,
                  usecols=["plasmid_id", "organism"]).set_index("plasmid_id")["organism"]

CAR = {r: sorted(set(CL.loc[CL["rep"].eq(r), "pl"])) for r in (A, B)}
for r in (A, B):
    L = pd.Series([len(SEQ[m]) for m in CL.loc[CL["rep"].eq(r), "member"]])
    print(f"{LAB[r]}  {r}")
    print(f"   {int(CL['rep'].eq(r).sum()):,} proteins on {len(CAR[r]):,} plasmids | "
          f"representative {len(SEQ[r])} aa | members {L.min()}-{L.max()} aa "
          f"(median {L.median():.0f})")

## 1. One object, not two

If these were independent discoveries they would ride different plasmids.

In [ ]:
# Q1 — are these two separate discoveries, or one object seen twice? If they rode different
# plasmids they would be independent; if they ride the same ones they are a cassette.
sa, sb = set(CAR[A]), set(CAR[B])
print(f"ORF-A carriers {len(sa):,} | ORF-B carriers {len(sb):,} | shared {len(sa & sb):,}")
print(f"  {len(sa & sb) / len(sa) * 100:.0f}% of A's carriers also carry B; "
      f"{len(sa & sb) / len(sb) * 100:.0f}% of B's also carry A  (Jaccard "
      f"{len(sa & sb) / len(sa | sb):.2f})")

BOTH = sorted(sa & sb)
for lab, s in (("A only", sa - sb), ("both", sa & sb), ("B only", sb - sa)):
    md = MET.reindex(sorted(s))
    print(f"  {lab:<7} n={len(s):>4}  size median {md['size_bp'].median():,.0f} bp  "
          f"mobility {md['mob_mobility'].value_counts().head(2).to_dict()}")
assert len(sa - sb) + len(sa & sb) == len(sa) and len(sb - sa) + len(sa & sb) == len(sb)

# Position. cds_index counts translated CDS in file order, so the difference is a gene offset.
a = CL[CL["rep"].eq(A)][["pl", "idx"]].rename(columns={"idx": "iA"})
b = CL[CL["rep"].eq(B)][["pl", "idx"]].rename(columns={"idx": "iB"})
J = a.merge(b, on="pl")
J["delta"] = J["iB"] - J["iA"]

# CAREFUL WITH THE DENOMINATOR. This merge is a per-plasmid cross product, so a plasmid with two
# copies of an ORF contributes several rows: len(J) is A-B PAIRINGS, not carriers. Collapsing to
# the closest pair on each plasmid gives one row per carrier, which is the honest unit here.
JN = J.loc[J.groupby("pl")["delta"].apply(lambda s: s.abs().idxmin())]
extra = J.groupby("pl").size()
print(f"\n{len(J):,} A-B pairings over {J['pl'].nunique():,} carriers "
      f"({int((extra > 1).sum())} plasmids carry more than one copy of an ORF and so contribute "
      f"several pairings)")
print(f"collapsing to the closest pair per plasmid -> {len(JN):,} rows, one per carrier\n")

print("gene offset between them, one observation per carrier:")
print(JN["delta"].value_counts().sort_index().to_string())
print(f"  |offset| == 2 on {JN['delta'].abs().eq(2).mean() * 100:.0f}% of the "
      f"{len(JN):,} carriers — never adjacent, always exactly one gene apart.")
print(f"  (the same figure computed over pairings instead would be "
      f"{J['delta'].abs().eq(2).mean() * 100:.0f}%.)")
print("  The sign is only which way the replicon happened to be numbered.")

## 2. What the replicon looks like

The offset says the two ORFs are separated by exactly one gene. The obvious next question is
which gene.

In [ ]:
# Q2 — what is the gene between them, and what is the rest of the replicon? ~5 min: scans the
# PlasAnn shards for the shared carriers only.
# Everything below counts JN, one row per carrier, not J. Counting pairings would over-weight
# the 11 plasmids that carry a duplicate ORF.
cids = set(BOTH)
parts = []
for f in sorted(glob.glob(str(ROOT / "data" / "plasann_run" / "annot" / "*shard_*.tsv.gz"))):
    d = pd.read_csv(f, sep="\t", low_memory=False)
    d = d[d["plasmid_id"].isin(cids)]
    if len(d):
        parts.append(d)
CG = pd.concat(parts, ignore_index=True)
# cds_index counts translated CDS only, so the ordering must exclude Replicon/ORIV/MGE rows.
CDS = CG[CG["feature type"].eq("CDS")].copy()
CDS["ord"] = CDS.groupby("plasmid_id").cumcount() + 1
GENE = CDS.set_index(["plasmid_id", "ord"])["Gene Name"]
BP = CDS.assign(bp=CDS["End"] - CDS["Start"] + 1).set_index(["plasmid_id", "ord"])["bp"]

# Validation of the index mapping: the CDS at index iA must have the length of an A protein.
print("index check — CDS length at the claimed index vs the family's member lengths:")
for r, col, lab in ((A, "iA", "ORF-A"), (B, "iB", "ORF-B")):
    obs = pd.Series([BP.get((x.pl, getattr(x, col))) for x in JN.itertuples()]).median()
    mem = pd.Series([len(SEQ[m]) for m in CL.loc[CL["rep"].eq(r), "member"]]).median()
    print(f"  {lab}: CDS {obs:,.0f} bp vs {mem:.0f} aa protein -> {mem * 3 + 3:,.0f} bp expected"
          f"   {'OK' if abs(obs - (mem * 3 + 3)) < 10 else 'MISMATCH'}")

mid = []
for x in JN.itertuples():
    lo, hi = sorted((x.iA, x.iB))
    for k in range(lo + 1, hi):
        g = GENE.get((x.pl, k))
        if g is not None:
            mid.append(g)
MID = pd.Series(mid).value_counts()
print(f"\ngene sitting BETWEEN ORF-A and ORF-B "
      f"({len(mid):,} genes over {len(JN):,} carriers — carriers at |offset| > 2 contribute "
      f"more than one):")
print(MID.head(8).to_string())
print(f"  a mobilisation gene (moB/mobA) in "
      f"{MID.reindex(['moB', 'mobA']).sum() / len(mid) * 100:.0f}% of cases")

SIG = CDS.groupby("plasmid_id")["Gene Name"].apply(lambda s: " - ".join(s)).value_counts()
print(f"\ngene order on the shared carriers — top 6 of {len(SIG)} distinct arrangements:")
print(SIG.head(6).to_string())

named = CG[CG["Gene Name"].ne("ORF")]
print(f"\nwhat else is on these {CG['plasmid_id'].nunique():,} plasmids "
      f"(by plasmids carrying the gene):")
print(named.groupby("Gene Name")["plasmid_id"].nunique().sort_values(ascending=False)
      .head(10).to_string())
print(f"\nmedian {CDS.groupby('plasmid_id').size().median():.0f} CDS per plasmid; "
      f"{CDS['Gene Name'].eq('ORF').mean() * 100:.0f}% of CDS are unannotatable ORFs")

In [ ]:
# Figure 1 — the cassette, drawn to scale. Colour is fixed for the whole figure: slot 1 = ORF-A,
# slot 2 = ORF-B, slot 3 = the mobilisation gene, grey = any other annotated gene. The offset
# statistics get their own figure so those four meanings are not reused.
MODAL_ID = (CDS.groupby("plasmid_id")["Gene Name"].apply(lambda s: " - ".join(s))
            .pipe(lambda s: s[s.eq(SIG.index[0])].index[0]))
ex = CDS[CDS["plasmid_id"].eq(MODAL_ID)].copy()
ex["ord"] = range(1, len(ex) + 1)
iA = int(JN.loc[JN["pl"].eq(MODAL_ID), "iA"].iloc[0])
iB = int(JN.loc[JN["pl"].eq(MODAL_ID), "iB"].iloc[0])


def gcol(row):
    if row["ord"] == iA:
        return S1
    if row["ord"] == iB:
        return S2
    return S3 if row["Gene Name"] in ("moB", "mobA") else NEUT


fig, ax = plt.subplots(figsize=(14, 2.6))
span = ex["End"].max()
for _, r in ex.iterrows():
    w = r["End"] - r["Start"]
    ax.arrow(r["Start"] if r["Strand"] == "+" else r["End"], 0,
             w if r["Strand"] == "+" else -w, 0, width=0.30, head_width=0.46,
             head_length=span * 0.016, length_includes_head=True, color=gcol(r), linewidth=0)
    nm = "ORF-A" if r["ord"] == iA else "ORF-B" if r["ord"] == iB else r["Gene Name"]
    ax.text((r["Start"] + r["End"]) / 2, 0.46, nm, ha="center", fontsize=9,
            color=INK if nm.startswith("ORF-") else INK2)
ax.set(xlim=(-span * 0.02, span * 1.02), ylim=(-0.75, 0.95), yticks=[])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlabel("position (bp)", fontsize=9, color=INK2)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set(color=GRID, linewidth=1)
ax.tick_params(colors=INK2, labelsize=9, length=0)
ax.set_title(f"The cassette — modal architecture drawn to scale\n"
             f"{MODAL_ID}, {MET.loc[MODAL_ID, 'size_bp']:,.0f} bp · this exact gene order on "
             f"{SIG.iloc[0]} of {len(BOTH)} carriers\n"
             f"blue = ORF-A · orange = ORF-B · green = mobilisation · grey = other annotated gene",
             fontsize=11, color=INK, loc="left")
plt.show()

In [ ]:
# Figure 2 — the two positional statistics. Grey means "everything else" in both panels; the
# highlight colour differs per panel because the panels highlight different things.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 4.2),
                               gridspec_kw={"wspace": 0.3, "width_ratios": [1.3, 1]})
# JN, not J: one row per carrier (see the denominator note in section 1).
d = JN["delta"].value_counts().sort_index()
d = d[d.index.to_series().abs() <= 5]
axA.bar(d.index, d.values, width=0.55, color=[S1 if abs(k) == 2 else NEUT for k in d.index])
for k, v in d.items():
    axA.text(k, v + d.max() * 0.02, f"{v}", ha="center", fontsize=8.5, color=INK2)
axA.set(xticks=d.index, yticks=[], ylim=(0, d.max() * 1.2))
axA.set_xlabel("gene offset (ORF-B index − ORF-A index)", fontsize=9, color=INK2)
axA.set_title(f"A — always exactly one gene apart\n|offset| = 2 on "
              f"{JN['delta'].abs().eq(2).mean() * 100:.0f}% of the {len(JN):,} carriers; the sign is only "
              f"which way the replicon was numbered", fontsize=10.5, color=INK, loc="left")
tidy(axA)

m = MID.head(5).iloc[::-1]
yb = np.arange(len(m))
axB.barh(yb, m.values, height=0.5,
         color=[S3 if k in ("moB", "mobA") else NEUT for k in m.index])
for i, v in enumerate(m.values):
    axB.text(v + m.max() * 0.02, i, f"{v}", va="center", fontsize=8.5, color=INK2)
axB.set(yticks=yb, yticklabels=m.index, xticks=[], xlim=(0, m.max() * 1.25))
axB.set_xlabel("times seen between the two ORFs", fontsize=9, color=INK2)
axB.set_title(f"B — and what sits between them\ngreen = mobilisation gene, "
              f"{MID.reindex(['moB', 'mobA']).sum() / MID.sum() * 100:.0f}% of cases",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("The spacing is fixed, and the gene between them is the relaxase",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.03)
plt.show()

## 3. Who carries it, and where

This is what made the pair stand out in the first place: 21–23 habitats on a handful of MOB
lineages.

In [ ]:
# Q3 — who carries this cassette, and where? Habitat spread is what made these two stand out in
# dark_families_90pct.ipynb (23 and 21 habitats), so the ecology is the point.
CB = MET.reindex(BOTH)
og = ORG.reindex(BOTH)
print(f"the {len(BOTH):,} plasmids carrying both ORFs:")
print(f"  size          median {CB['size_bp'].median():,.0f} bp  "
      f"IQR {CB['size_bp'].quantile(.25):,.0f}-{CB['size_bp'].quantile(.75):,.0f}")
print(f"  mobility      {CB['mob_mobility'].value_counts(dropna=False).to_dict()}")
print(f"  replicon      {CB['mob_rep_types'].value_counts(dropna=False).head(3).to_dict()}")
print(f"  MOB lineages  {CB['mob_cluster'].nunique()}  "
      f"(largest holds {CB['mob_cluster'].value_counts(normalize=True).iloc[0] * 100:.0f}%)")
print(f"  carries AMR   {(CB['card_n_arg'].fillna(0) > 0).sum()} "
      f"({(CB['card_n_arg'].fillna(0) > 0).mean() * 100:.0f}%)")
print(f"\nhost (named on {og.notna().sum()} of {len(BOTH)}):")
print(og.value_counts().head(8).to_string())
print(f"\nhabitat ({CB['hab_sub'].nunique()} distinct):")
print(CB["hab_sub"].value_counts().head(8).to_string())
print(f"\ncountry ({CB['geo_country'].nunique()} distinct, known for "
      f"{CB['geo_country'].notna().sum()}):")
print(CB["geo_country"].value_counts().head(8).to_string())

# The tension worth stating plainly, because it decides how the spread should be read.
print(f"\nSo: {og.nunique()} named host species across {CB['hab_sub'].nunique()} habitats and "
      f"{CB['geo_country'].nunique()} countries,\nbut only {CB['mob_cluster'].nunique()} MOB "
      f"primary clusters, the largest holding "
      f"{CB['mob_cluster'].value_counts(normalize=True).iloc[0] * 100:.0f}%.\n"
      f"Ecologically wide, genetically narrow — one plasmid backbone that has got everywhere,\n"
      f"not many backbones converging on the same two genes.")

In [ ]:
# Figure 2. Grey is unused as a category here; each panel is one series in slot 1, because the
# panels measure different things and no comparison is being drawn between them.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"wspace": 0.42})
for ax, s, ttl, n in (
        (axes[0], og.value_counts().head(8), "A — host species",
         f"named on {og.notna().sum()} of {len(BOTH)} carriers"),
        (axes[1], CB["hab_sub"].value_counts().head(8), "B — habitat",
         f"{CB['hab_sub'].nunique()} distinct"),
        (axes[2], CB["geo_country"].value_counts().head(8), "C — country",
         f"{CB['geo_country'].nunique()} distinct")):
    t = s.iloc[::-1]
    y = np.arange(len(t))
    ax.barh(y, t.values, height=0.55, color=S1)
    for i, v in enumerate(t.values):
        ax.text(v + t.max() * 0.03, i, f"{v}", va="center", fontsize=8.5, color=INK2)
    ax.set(yticks=y, yticklabels=[str(k)[:26] for k in t.index], xticks=[],
           xlim=(0, t.max() * 1.22))
    ax.tick_params(axis="y", labelsize=8)
    ax.set_xlabel("plasmids", fontsize=9, color=INK2)
    ax.set_title(f"{ttl}\n{n}", fontsize=10.5, color=INK, loc="left")
    tidy(ax)
fig.suptitle(f"Where the cassette lives — {len(BOTH):,} plasmids carrying both ORFs",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.04)
plt.show()

## 4. Two loose ends

Does the cassette reach large plasmids, and how conserved is each protein inside its family?

In [ ]:
# Q4 — two loose ends. Does the cassette reach large plasmids, and how conserved is each ORF?
hits = pd.read_csv(ROOT / "data" / "zoom_two_orfs" / "big_hits.tsv", sep="\t", header=None,
                   names=["query", "target", "fident", "qcov", "tcov", "evalue"])
hits["pl"] = hits["target"].str.rsplit("|", n=1).str[0]
print("searched against the dark proteome of all 61,242 plasmids >= 20 kb")
print("(scripts/fam_bigplasmid_search.sbatch built that set; mmseqs easy-search at the same "
      "30% id / 80% cov)")
for q, g in hits.groupby("query"):
    print(f"  {LAB[q]}: {g['target'].nunique()} proteins on {g['pl'].nunique()} big plasmids")
print(f"  {hits['pl'].nunique()} distinct big plasmids in total — "
      f"{hits['pl'].nunique() / 61_242 * 100:.3f}% of them.")
print("  This cassette is a small-plasmid feature, not a general mobile element.")

print("\nsequence conservation within each family (90% identity clustering, so this is the")
print("residual variation inside an already-tight cluster):")
for r in (A, B):
    mem = CL.loc[CL["rep"].eq(r), "member"]
    L = pd.Series([len(SEQ[m]) for m in mem])
    uniq = len({SEQ[m] for m in mem})
    print(f"  {LAB[r]}: {len(mem)} proteins, {uniq} distinct sequences "
          f"({(1 - uniq / len(mem)) * 100:.0f}% identical duplicates), "
          f"lengths {L.min()}-{L.max()} aa "
          f"({(L == L.mode()[0]).mean() * 100:.0f}% are exactly {L.mode()[0]} aa)")

print("\nthe two sequences (family representatives):")
for r in (A, B):
    print(f"\n>{LAB[r]}  {r}  ({len(SEQ[r])} aa)")
    for i in range(0, len(SEQ[r]), 60):
        print("  " + SEQ[r][i:i + 60])

OUT = ROOT / "data" / "zoom_two_orfs"
OUT.mkdir(exist_ok=True)
pd.DataFrame({"plasmid_id": BOTH}).join(
    MET.reindex(BOTH)[["size_bp", "mob_mobility", "mob_cluster", "hab_sub",
                       "geo_country"]].reset_index(drop=True)).to_csv(
    OUT / "cassette_carriers.tsv", sep="\t", index=False)
print(f"\ncarrier table -> {(OUT / 'cassette_carriers.tsv').relative_to(ROOT)} ({len(BOTH)} rows)")

## 5. What the sequences themselves suggest

No structure prediction has been run. Amino-acid composition is the only structural read
available, and it is a hypothesis generator rather than an answer.

In [ ]:
# Q6 — the only structural read available without folding: amino-acid composition. Kyte-Doolittle
# hydropathy over a 19-residue window is the classic transmembrane screen; the conventional
# threshold is a window mean above ~1.6. This is a hypothesis generator, not a prediction.
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5, "G": -0.4,
      "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, "S": -0.8,
      "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
CHG = {"K": 1, "R": 1, "H": 0.1, "D": -1, "E": -1}
W = 19

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"wspace": 0.22})
for ax, r in zip(axes, (A, B)):
    s = SEQ[r]
    h = np.array([np.mean([KD.get(c, 0) for c in s[i:i + W]]) for i in range(len(s) - W + 1)])
    x = np.arange(len(h)) + W // 2
    ax.axhline(1.6, color=S2, linewidth=1.2, zorder=1)
    ax.text(1, 1.75, "1.6 — transmembrane threshold", fontsize=8, color=S2)
    ax.axhline(0, color=GRID, linewidth=1, zorder=0)
    ax.plot(x, h, color=S1, linewidth=1.8, zorder=2)
    ax.fill_between(x, 1.6, h, where=h > 1.6, color=S2, alpha=0.30, linewidth=0, zorder=1)
    net = sum(CHG.get(c, 0) for c in s)
    over = int((h > 1.6).sum())
    ax.set(xlim=(0, len(s)), ylim=(min(-3.2, h.min() - 0.4), max(2.6, h.max() + 0.4)))
    ax.set_xlabel("residue", fontsize=9, color=INK2)
    ax.set_ylabel("Kyte-Doolittle (19 aa window)", fontsize=9, color=INK2)
    # Describe where the segment is rather than asserting it; the range is computed.
    if over:
        hot = x[h > 1.6]
        where = f"one hydrophobic segment, residues {hot.min()}-{hot.max()} of {len(s)}"
    else:
        where = "no window reaches the threshold — likely soluble"
    ax.set_title(f"{LAB[r]} — {len(s)} aa · net charge {net:+.0f} · "
                 f"{over} windows above threshold\n{where}",
                 fontsize=10.5, color=INK, loc="left")
    tidy(ax)
fig.suptitle("Composition of the two proteins — orange marks hydrophobic stretches",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.06)
plt.show()

for r in (A, B):
    s = SEQ[r]
    h = np.array([np.mean([KD.get(c, 0) for c in s[i:i + W]]) for i in range(len(s) - W + 1)])
    net = sum(CHG.get(c, 0) for c in s)
    basic = sum(c in "KR" for c in s) / len(s) * 100
    print(f"{LAB[r]}: {len(s)} aa | net charge {net:+.0f} | {basic:.0f}% K/R | "
          f"max hydropathy {h.max():.2f} | "
          f"{'has' if (h > 1.6).any() else 'no'} predicted TM segment")
print("\nRead with care: hydropathy is suggestive, not a structure. Both proteins are in\n"
      "data/zoom_two_orfs/ ready for folding, which is the test that would settle it.")

## 6. A threshold artifact worth knowing about

ORF-A is a 20-protein family at 30% identity and a 400-plasmid family at 90%. That is not a
mistake in either run, and the reason generalises.

In [ ]:
# Q5 — the threshold artifact worth knowing about. ORF-A looks like a trivial 20-protein family
# at 30% identity and a 400-plasmid family at 90%. Families are NOT nested across thresholds,
# and the reason is the coverage rule interacting with which sequence becomes representative.
print("family containing ORF-A at each threshold in the sweep:")
prev = None
for tag in ("dark30", "dark50", "dark70", "dark90"):
    c = pd.read_csv(RC / f"{tag}_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
    fr = c.set_index("member")["rep"].get(A)
    mem = set(c.loc[c["rep"].eq(fr), "member"])
    nest = "" if prev is None else ("  (subset of the looser family: "
                                    f"{'yes' if mem <= prev[1] else 'NO'})")
    print(f"  {tag}: representative {fr:<40} {len(mem):>4} proteins{nest}")
    prev = (tag, mem)

c30 = pd.read_csv(RC / "dark30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
c70 = pd.read_csv(RC / "dark70_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
m2r30 = c30.set_index("member")["rep"]
fam70 = set(c70.loc[c70["rep"].eq(c70.set_index("member")["rep"].get(A)), "member"])
homes = pd.Series([m2r30.get(m) for m in fam70]).value_counts()
f30 = pd.read_csv(RC / "dark30_fampfam.tsv", sep="\t").set_index("rep")
print(f"\nthe {len(fam70)} proteins that form ONE family at 70% were split across "
      f"{len(homes)} families at 30%:")
for rep, n in homes.items():
    print(f"  {n:>4} of them in {rep}  ({f30.loc[rep, 'rep_len_aa']:.0f} aa representative)")
print(f"\nWhy: -c 0.8 --cov-mode 0 requires 80% coverage of BOTH sequences, so a "
      f"{f30.loc[homes.index[0], 'rep_len_aa']:.0f} aa\nrepresentative and a {len(SEQ[A])} aa "
      f"protein can never share a family at any identity "
      f"({len(SEQ[A]) / f30.loc[homes.index[0], 'rep_len_aa']:.2f} < 0.80).")
print("At 30% greedy set-cover picks the long protein as a hub and absorbs everything it covers;\n"
      "the shorter ones are left in a small cluster. At 70% the hub loses its long-range links\n"
      "and the short proteins coalesce into their own large family.")
print("\nConsequence: a family being small at a loose threshold does NOT mean it is unimportant.\n"
      "The 30% view understated ORF-A twentyfold.")

## What these two are

**They are one cassette, not two findings.** 336 plasmids carry both — 84% of ORF-A's 400
carriers and 95% of ORF-B's 355, Jaccard 0.80. Every one of those 336 is `mobilizable`, median
5,192 bp.

**They flank the relaxase.** The offset between them is exactly ±2 genes on **87% of the 336
carriers** — never adjacent, always one gene apart — and that gene is a mobilisation gene
(`moB` or `mobA`) in **72%** of cases. The modal architecture is

```
reP  —  ORF-B  —  moB  —  ORF-A  —  yibT
```

on a **Col156** replicon (333 of 336 carriers). These are textbook small ColE1-family
mobilizable plasmids, and **48% of their CDS are unannotatable**. Two of the five or six genes
on the canonical ColE1-like backbone have no name.

> **A denominator that is easy to get wrong.** Merging the two ORFs by plasmid gives **365 rows
> over 336 carriers**, because 11 plasmids carry a duplicate copy of one ORF and so contribute
> several A–B pairings. Percentages have to be taken per carrier, not per pairing: the offset
> statistic is 87% per carrier against 81% per pairing, and the mobilisation-gene share is 72%
> against 61%. Section 1 collapses to the closest pair on each plasmid before counting.

**Ecologically wide, genetically narrow.** 26 named host species (mostly *E. coli*, with
*Klebsiella*, *Salmonella*, *Shigella*, *Enterobacter*), 21 habitats — human gut, blood, urine,
wastewater, poultry — and 30 countries. But only **4 MOB primary clusters, the largest holding
78%**. So the breadth is one backbone that has travelled, not many backbones converging. That
distinction matters: it is a dissemination result, not a convergent-evolution result.

**Not a general mobile element.** Searched against the dark proteome of all 61,242 plasmids
≥ 20 kb, the pair hits **3 big plasmids between them** — 0.005%. This is a small-plasmid
feature.

**And they are almost invariant.** Inside each 90% family, 93% of proteins are exact duplicates
of another member; ORF-B is **exactly 132 aa in all 365 copies**, ORF-A exactly 177 aa in 98% of
its 411. Sequence that fixed across 30 countries and 21 habitats is under selection.

**Zero AMR.** None of the 336 carriers has a CARD hit. Whatever these genes do, it is not
resistance.

### What they might be

ORF-A carries a single hydrophobic segment above the transmembrane threshold and a net charge
of +2; ORF-B has no hydrophobic window at all, a net charge of +6 and 20% K/R. So the pair
looks like *one membrane-associated protein and one basic soluble protein, flanking a
relaxase* — which is the arrangement of a mobilisation accessory module, and would also fit
entry exclusion or a small regulatory system. That is a hypothesis from composition alone.
The sequences are in `data/zoom_two_orfs/two_reps.faa`; folding plus a structural search is
the test that would settle it.

### The methodological point

ORF-A is a 20-protein family at 30% identity and a 400-plasmid family at 90%. The clusterings
are **not nested**, because `-c 0.8 --cov-mode 0` requires 80% coverage of both sequences: a
208 aa representative and a 155 aa protein can never share a family at any identity
(0.75 < 0.80). At 30% greedy set-cover makes the long protein a hub that absorbs everything it
covers, stranding the short ones; at 70% that hub loses its long-range links and the short
proteins coalesce.

So a family being small at a loose threshold does **not** mean it is unimportant. The 30% view
understated ORF-A twentyfold, and it would have been missed entirely by ranking families at
that threshold — which is exactly how `dark_families.ipynb` built its shortlist.